In [ ]:
import pandas as pd
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import re

In [ ]:
a1 = pd.read_csv('/content/a1_RestaurantReviews_HistoricDump.tsv', sep = '\t')
a1

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1
...,...,...
895,I want to first say our server was great and w...,1
896,The pizza selections are good.,1
897,"I had strawberry tea, which was good.",1
898,Highly unprofessional and rude to a loyal patron!,0


# **Text Blob**

In [ ]:
!pip install textblob

In [ ]:
from textblob import TextBlob

a1['TextBlob_Polarity'] = a1['Review'].apply(
    lambda x: TextBlob(str(x)).sentiment.polarity
)

a1['TextBlob_Subjectivity'] = a1['Review'].apply(
    lambda x: TextBlob(str(x)).sentiment.subjectivity
)

In [ ]:
def textblob_sentiment(score):
    if score > 0:
        return 'Positive'
    elif score < 0:
        return 'Negative'
    else:
        return 'Neutral'

a1['TextBlob_Sentiment'] = a1['TextBlob_Polarity'].apply(textblob_sentiment)


<function textblob_sentiment at 0x7e40059f2b60>


In [ ]:
a1[['Review', 'Liked', 'TextBlob_Polarity',
    'TextBlob_Subjectivity', 'TextBlob_Sentiment']].head(20)

,Review,Liked,TextBlob_Polarity,TextBlob_Subjectivity,TextBlob_Sentiment
0,Wow... Loved this place.,1,0.40000,0.900000,Positive
1,Crust is not good.,0,-0.35000,0.600000,Negative
2,Not tasty and the texture was just nasty.,0,-1.00000,1.000000,Negative
3,Stopped by during the late May bank holiday of...,1,0.20000,0.700000,Positive
4,The selection on the menu was great and so wer...,1,0.80000,0.750000,Positive
5,Now I am getting angry and I want my damn pho.,0,-0.50000,1.000000,Negative
6,Honeslty it didn't taste THAT fresh.),0,0.30000,0.500000,Positive
7,The potatoes were like rubber and you could te...,0,0.00000,0.000000,Neutral
8,The fries were great too.,1,0.80000,0.750000,Positive
9,A great touch.,1,0.80000,0.750000,Positive


# **Sentiment Analyzer**

In [ ]:
import nltk

nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

a1['VADER_Score'] = a1['Review'].apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

def vader_sentiment(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

a1['VADER_Sentiment'] = a1['VADER_Score'].apply(vader_sentiment)

In [ ]:
a1[['Review', 'Liked', 'VADER_Score', 'VADER_Sentiment']].head(20)

,Review,Liked,VADER_Score,VADER_Sentiment
0,Wow... Loved this place.,1,0.5994,Positive
1,Crust is not good.,0,-0.3412,Negative
2,Not tasty and the texture was just nasty.,0,-0.5574,Negative
3,Stopped by during the late May bank holiday of...,1,0.6908,Positive
4,The selection on the menu was great and so wer...,1,0.6249,Positive
5,Now I am getting angry and I want my damn pho.,0,-0.6908,Negative
6,Honeslty it didn't taste THAT fresh.),0,0.0000,Neutral
7,The potatoes were like rubber and you could te...,0,0.5719,Positive
8,The fries were great too.,1,0.6249,Positive
9,A great touch.,1,0.6249,Positive


# **Hugging Face**

In [ ]:
!pip install transformers torch

from transformers import pipeline

In [ ]:
hf_sentiment = pipeline("sentiment-analysis")

review = a1['Review'][0]

result = hf_sentiment(review)

print(review)
print(result)

a1['HF_Result'] = a1['Review'].head(10).apply(
    lambda x: hf_sentiment(str(x))[0]
)

a1[['Review', 'Liked', 'HF_Result']].head(10)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Wow... Loved this place.
[{'label': 'POSITIVE', 'score': 0.9998804330825806}]


,Review,Liked,HF_Result
0,Wow... Loved this place.,1,"{'label': 'POSITIVE', 'score': 0.9998804330825..."
1,Crust is not good.,0,"{'label': 'NEGATIVE', 'score': 0.9997690320014..."
2,Not tasty and the texture was just nasty.,0,"{'label': 'NEGATIVE', 'score': 0.9996275901794..."
3,Stopped by during the late May bank holiday of...,1,"{'label': 'POSITIVE', 'score': 0.9995303153991..."
4,The selection on the menu was great and so wer...,1,"{'label': 'POSITIVE', 'score': 0.9996086955070..."
5,Now I am getting angry and I want my damn pho.,0,"{'label': 'NEGATIVE', 'score': 0.992295503616333}"
6,Honeslty it didn't taste THAT fresh.),0,"{'label': 'NEGATIVE', 'score': 0.9909782409667..."
7,The potatoes were like rubber and you could te...,0,"{'label': 'NEGATIVE', 'score': 0.9024147391319..."
8,The fries were great too.,1,"{'label': 'POSITIVE', 'score': 0.9997226595878..."
9,A great touch.,1,"{'label': 'POSITIVE', 'score': 0.9998835325241..."


In [ ]:
hf_results = hf_sentiment(
    a1['Review'].astype(str).tolist(),
    truncation=True
)

a1['HF_Sentiment'] = [result['label'] for result in hf_results]

a1['HF_Score'] = [result['score'] for result in hf_results]

a1[['Review', 'Liked', 'HF_Sentiment', 'HF_Score']].head(20)

,Review,Liked,HF_Sentiment,HF_Score
0,Wow... Loved this place.,1,POSITIVE,0.999880
1,Crust is not good.,0,NEGATIVE,0.999769
2,Not tasty and the texture was just nasty.,0,NEGATIVE,0.999628
3,Stopped by during the late May bank holiday of...,1,POSITIVE,0.999530
4,The selection on the menu was great and so wer...,1,POSITIVE,0.999609
5,Now I am getting angry and I want my damn pho.,0,NEGATIVE,0.992296
6,Honeslty it didn't taste THAT fresh.),0,NEGATIVE,0.990978
7,The potatoes were like rubber and you could te...,0,NEGATIVE,0.902415
8,The fries were great too.,1,POSITIVE,0.999723
9,A great touch.,1,POSITIVE,0.999884


In [ ]:
a1[['Review',
    'Liked',
    'TextBlob_Sentiment',
    'VADER_Sentiment',
    'HF_Sentiment']].head(20)

,Review,Liked,TextBlob_Sentiment,VADER_Sentiment,HF_Sentiment
0,Wow... Loved this place.,1,Positive,Positive,POSITIVE
1,Crust is not good.,0,Negative,Negative,NEGATIVE
2,Not tasty and the texture was just nasty.,0,Negative,Negative,NEGATIVE
3,Stopped by during the late May bank holiday of...,1,Positive,Positive,POSITIVE
4,The selection on the menu was great and so wer...,1,Positive,Positive,POSITIVE
5,Now I am getting angry and I want my damn pho.,0,Negative,Negative,NEGATIVE
6,Honeslty it didn't taste THAT fresh.),0,Positive,Neutral,NEGATIVE
7,The potatoes were like rubber and you could te...,0,Neutral,Positive,NEGATIVE
8,The fries were great too.,1,Positive,Positive,POSITIVE
9,A great touch.,1,Positive,Positive,POSITIVE


In [ ]:
ps = PorterStemmer()

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
corpus = []
for i in range(0, len(a1)):
  review = re.sub('[^a-zA-Z]', ' ', a1['Review'][i])
  review = review.lower()
  review = review.split()
  print(review)
  review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
  review = ' '.join(review)
  print(review)
  corpus.append(review)

['wow', 'loved', 'this', 'place']
wow love place
['crust', 'is', 'not', 'good']
crust good
['not', 'tasty', 'and', 'the', 'texture', 'was', 'just', 'nasty']
tasti textur nasti
['stopped', 'by', 'during', 'the', 'late', 'may', 'bank', 'holiday', 'off', 'rick', 'steve', 'recommendation', 'and', 'loved', 'it']
stop late may bank holiday rick steve recommend love
['the', 'selection', 'on', 'the', 'menu', 'was', 'great', 'and', 'so', 'were', 'the', 'prices']
select menu great price
['now', 'i', 'am', 'getting', 'angry', 'and', 'i', 'want', 'my', 'damn', 'pho']
get angri want damn pho
['honeslty', 'it', 'didn', 't', 'taste', 'that', 'fresh']
honeslti tast fresh
['the', 'potatoes', 'were', 'like', 'rubber', 'and', 'you', 'could', 'tell', 'they', 'had', 'been', 'made', 'up', 'ahead', 'of', 'time', 'being', 'kept', 'under', 'a', 'warmer']
potato like rubber could tell made ahead time kept warmer
['the', 'fries', 'were', 'great', 'too']
fri great
['a', 'great', 'touch']
great touch
['service', '

In [ ]:
print(review)
print(corpus)

overal great experi
['wow love place', 'crust good', 'tasti textur nasti', 'stop late may bank holiday rick steve recommend love', 'select menu great price', 'get angri want damn pho', 'honeslti tast fresh', 'potato like rubber could tell made ahead time kept warmer', 'fri great', 'great touch', 'servic prompt', 'would go back', 'cashier care ever say still end wayyy overpr', 'tri cape cod ravoli chicken cranberri mmmm', 'disgust pretti sure human hair', 'shock sign indic cash', 'highli recommend', 'waitress littl slow servic', 'place worth time let alon vega', 'like', 'burritto blah', 'food amaz', 'servic also cute', 'could care less interior beauti', 'perform', 'right red velvet cake ohhh stuff good', 'never brought salad ask', 'hole wall great mexican street taco friendli staff', 'took hour get food tabl restaur food luke warm sever run around like total overwhelm', 'worst salmon sashimi', 'also combo like burger fri beer decent deal', 'like final blow', 'found place accid could hap

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(binary = True, max_features=1420)
x = cv.fit_transform(corpus)
#x[0].toarray()
x

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 4771 stored elements and shape (900, 1420)>

In [ ]:
y = a1['Liked']
y = a1.iloc[:, -1].values
y

array([1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1,
       1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0,
       0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1,

In [ ]:
import pickle
bow_path = 'c1.pkl'
pickle.dump(cv, open(bow_path, 'wb'))

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=0)

In [ ]:
from sklearn.naive_bayes import MultinomialNB #2nd type gaussian
reviews_n_liked = MultinomialNB().fit(x_train, y_train)

In [ ]:
y_pred = reviews_n_liked.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

score = accuracy_score(y_test, y_pred)
print(score)

cm = confusion_matrix(y_test, y_pred)
print(cm)

cr = classification_report(y_test, y_pred)
print(cr)

0.7388888888888889
[[51 27]
 [20 82]]
              precision    recall  f1-score   support

           0       0.72      0.65      0.68        78
           1       0.75      0.80      0.78       102

    accuracy                           0.74       180
   macro avg       0.74      0.73      0.73       180
weighted avg       0.74      0.74      0.74       180



In [ ]:
#kappa Score
from sklearn.metrics import cohen_kappa_score
print("Kappa Score:", cohen_kappa_score(y_test, y_pred))

Kappa Score: 0.46265243902439024


In [ ]:
import joblib
joblib.dump(reviews_n_liked, 'c2_reviews_n_liked_model')

['c2_reviews_n_liked_model']

In [ ]:
a1['TextBlob_Prediction'] = a1['TextBlob_Sentiment'].map({
    'Positive': 1,
    'Negative': 0,
    'Neutral': 0
})

In [ ]:
from sklearn.metrics import accuracy_score

textblob_accuracy = accuracy_score(
    a1['Liked'],
    a1['TextBlob_Prediction']
)

print("TextBlob Accuracy:", textblob_accuracy)

TextBlob Accuracy: 0.7755555555555556


In [ ]:
a1['VADER_Prediction'] = a1['VADER_Sentiment'].map({
    'Positive': 1,
    'Negative': 0,
    'Neutral': 0
})

vader_accuracy = accuracy_score(
    a1['Liked'],
    a1['VADER_Prediction']
)

print("VADER Accuracy:", vader_accuracy)

VADER Accuracy: 0.8133333333333334


In [ ]:
a1['HF_Prediction'] = a1['HF_Sentiment'].map({
    'POSITIVE': 1,
    'NEGATIVE': 0
})

hf_accuracy = accuracy_score(
    a1['Liked'],
    a1['HF_Prediction']
)

print("Hugging Face Accuracy:", hf_accuracy)

Hugging Face Accuracy: 0.9244444444444444


In [ ]:
print("TextBlob Accuracy:", textblob_accuracy)
print("VADER Accuracy:", vader_accuracy)
print("Hugging Face Accuracy:", hf_accuracy)

TextBlob Accuracy: 0.7755555555555556
VADER Accuracy: 0.8133333333333334
Hugging Face Accuracy: 0.9244444444444444
